# 01 — Baseline Reproduction (B0–B3)

**Purpose.** Loads the four reference baselines used throughout this thesis — plain
ResNet-50 (B0), plain AlexNet (B1), VOneResNet-50 (B2), and standalone CORnet-S (B3) —
and verifies their relative robustness ordering on one evaluation slice (clean accuracy
+ a bounded PGD attack), reproducing the qualitative finding behind Bulgu 1 in the
thesis: a fixed, biologically-constrained V1 front-end (B2) raises adversarial
robustness over a plain CNN (B0) without adversarial training.

**Math covered in this notebook.**
- The VOneBlock linear-nonlinear-Poisson (LNP) model: Gabor filter bank, simple/complex
  cell nonlinearities, neuronal stochasticity — matched line-by-line against
  `external/vonenet/vonenet/modules.py`.
- The PGD attack and its Expectation-over-Transformation (EOT) extension, required for
  a fair evaluation of the stochastic VOneResNet-50 baseline.
- Liao & Poggio's (2016) architectural argument for treating a residual network as an
  unrolled recurrent computation — the justification for using ResNet-50 (rather than
  CORnet-S) as this thesis's main ventral-stream backbone (see docx Sec. 7, point 3).

**Ablation IDs covered.** B0, B1, B2, B3 (baseline reproduction only — no ablation flags
are exercised yet; those start in `02_foveation_rblur_and_periphery.ipynb` and
`03_v1_block.ipynb`).

**Inputs.** `external/vonenet`, `external/CORnet` (cloned by notebook 00),
`checkpoints/00_setup_complete.pt`.

**Outputs.** `results/01_baseline_metrics.json`, `results/01_baseline_comparison.png`,
`results/01_environment.json`, `checkpoints/01_baselines_evaluated.pt`.


## Binding rules (recap)

Same three rules as every notebook in this project (see `00_setup_and_data.ipynb` Sec.
"Binding rules" and `TEKNIK_REHBER.md` for the full rationale): Colab-first, notebooks
as the unit of work with shared code in `src/`, and never edit `external/` — only
subclass, monkey-patch, or wrap.


In [ ]:
# --- Environment detection: Colab or local? ---
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    PROJECT_ROOT = os.path.expanduser("~/Desktop/Bogazici/Tez/foveated_vision")

EXTERNAL_DIR = os.path.join(PROJECT_ROOT, "external")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CKPT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
for d in (PROJECT_ROOT, EXTERNAL_DIR, SRC_DIR, DATA_DIR, CKPT_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

import torch
print(f"Running in Colab: {IN_COLAB}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Pinned dependencies (avoids breakage from Colab image drift). Safe to re-run.
!pip -q install torch==2.* torchvision timm==1.* foolbox==3.* robustbench \
    scipy scikit-learn matplotlib tqdm requests


In [ ]:
# Make `src/` importable, and ensure the repos this notebook needs are present
# (idempotent -- see 00_setup_and_data.ipynb for the full explanation of the override
# policy; repeated here only so this notebook can also run standalone).
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.common as common
import src.datasets as datasets
import src.eval_harness as eval_harness
import src.models as models

REPOS = {
    "vonenet": "https://github.com/dicarlolab/vonenet.git",
    "CORnet": "https://github.com/dicarlolab/CORnet.git",
    "fovea": "https://github.com/tchittesh/fovea.git",
}
for name, url in REPOS.items():
    dst = os.path.join(EXTERNAL_DIR, name)
    if not os.path.exists(dst):
        print(f"Cloning {name} ...")
        subprocess.run(["git", "clone", "--depth", "1", url, dst], check=True)
    if dst not in sys.path:
        sys.path.insert(0, dst)

setup_ckpt = common.load_checkpoint(os.path.join(CKPT_DIR, "00_setup_complete.pt"))
if setup_ckpt is None:
    print("WARNING: checkpoints/00_setup_complete.pt not found -- "
          "00_setup_and_data.ipynb has not been run in this environment yet. "
          "Continuing, but dataset paths may be missing.")
else:
    print("Found 00_setup_complete.pt:", setup_ckpt)

print("Imported src.common, src.datasets, src.eval_harness, src.models successfully.")


## Configuration and seeding

Two knobs matter here beyond the usual `smoke_test`:
- `pretrained = not CFG['smoke_test']`: in a smoke test, every baseline is built with
  fresh random weights and **no network access** (fast, deterministic pipeline check).
  In a full run, real pretrained weights are downloaded for all four baselines. Any
  accuracy computed under `pretrained=False` is meaningless by construction and must
  never be reported as a result.
- `pgd_steps` / `max_samples` are much smaller under `smoke_test` — this notebook is
  meant to *exercise the code path*, not to produce the thesis's final robustness
  numbers (those come from the full ablation sweep in
  `06_full_model_and_ablations.ipynb`).


In [ ]:
CFG = {
    "seed": 0,
    "smoke_test": True,
    "image_size": 224,
    "data_dir": DATA_DIR,
    "batch_size": 4,
    # PGD attack budget, in raw [0, 1] pixel units (standard ImageNet convention).
    "pgd_eps": 8 / 255,
    "pgd_alpha": 2 / 255,
}
CFG["pretrained"] = not CFG["smoke_test"]
CFG["pgd_steps"] = 3 if CFG["smoke_test"] else 10
CFG["eot_samples"] = 2 if CFG["smoke_test"] else 10   # only used for the stochastic B2
CFG["max_samples"] = 8 if CFG["smoke_test"] else 200  # "one robustness slice", not full val

common.set_seed(CFG["seed"])
print("CFG:", CFG)


## Math core 1: the VOneBlock LNP model

VOneResNet-50 (baseline B2) prepends a fixed-weight **linear-nonlinear-Poisson (LNP)**
model of primate V1 — the *VOneBlock* — to a standard ResNet-50 back-end (Dapello et
al., 2020). This section derives its three stages exactly as implemented in
`external/vonenet/vonenet/modules.py` and `utils.py`, so that the code below can be
read as a direct application of these equations rather than a black box.

### Stage 1 — Gabor filter bank

Each of the $C = {}$`simple_channels` $+$ `complex_channels` output channels has an
associated 2D Gabor kernel (`utils.py::gabor_kernel`), built from a rotated coordinate
frame

$$
x' = x\cos\theta + y\sin\theta, \qquad y' = -x\sin\theta + y\cos\theta,
$$

as a Gaussian-windowed sinusoid

$$
G(x, y) \;=\; \frac{1}{2\pi\sigma_x\sigma_y}\,
\exp\!\Big(-\tfrac12\big(\tfrac{x'^2}{\sigma_x^2} + \tfrac{y'^2}{\sigma_y^2}\big)\Big)\,
\cos\!\big(2\pi f\, x' + \phi\big).
$$

Two filter banks are built per channel, in *quadrature* (90° phase offset):
$q_0$ uses phase $\phi_i$, $q_1$ uses phase $\phi_i + \pi/2$
(`simple_conv_q1.initialize(..., phase=self.phase + np.pi/2)`). Orientation $\theta$,
spatial frequency $f$, and envelope shape $(\sigma_x, \sigma_y)$ are drawn per-channel
from empirical macaque V1 tuning distributions (`params.py::generate_gabor_param`,
citing DeValois 1982, Ringach 2002, Schiller 1976), not learned.

### Stage 2 — Simple and complex cell nonlinearities

Convolving the image with both banks gives $s_{q0} = I * G_{q0}$, $s_{q1} = I * G_{q1}$.
The first $N_s = {}$`simple_channels` output channels are read out as **simple cells**
via half-wave rectification of the $q_0$ response only:

$$
r_s^{(i)} \;=\; k_{\text{exc}}\,\big[\,s_{q0}^{(i)}\,\big]_+, \qquad i < N_s,
$$

while the remaining channels are read out as **complex cells** via the *quadrature
energy* of the same two responses (phase-invariant, since $s_{q0}, s_{q1}$ are 90° apart):

$$
r_c^{(i)} \;=\; \frac{k_{\text{exc}}}{\sqrt2}\,
\sqrt{\big(s_{q0}^{(i)}\big)^2 + \big(s_{q1}^{(i)}\big)^2}, \qquad i \ge N_s.
$$

(`modules.py::gabors_f` computes exactly this: `s = ReLU(s_q0[:, :N_s])`,
`c = sqrt(s_q0[:, N_s:]**2 + s_q1[:, N_s:]**2) / sqrt(2)`, concatenated and scaled by
`k_exc`.)

### Stage 3 — Neuronal (Poisson-like) stochasticity

Let $r$ be the concatenated simple/complex response and $\kappa={}$`noise_scale`,
$\lambda={}$`noise_level`. `modules.py::noise_f` (mode `"neuronal"`, VOneResNet-50's
default) computes

$$
y = \kappa\, r + \lambda, \qquad
\tilde y = y + \xi\sqrt{[y]_+ + \epsilon}, \quad \xi \sim \mathcal N(0, 1),
$$
$$
r^{\text{VOne}} \;=\; \Big[\,\tfrac{\tilde y - \lambda}{\kappa}\,\Big]_+ .
$$

Because the injected noise's standard deviation scales with $\sqrt{[y]_+}$, its
*variance* scales linearly with the (rectified) mean response — the defining property
of a Poisson process, hence "Poisson-like." `fix_noise(seed=...)` pre-samples $\xi$ once
so evaluation is deterministic; `unfix_noise()` reverts to fresh $\xi$ every forward
pass, which is what an EOT attack (Math core 2) needs.


In [ ]:
# Ground the equations above in the real VOneBlock: build one (no pretrained
# weights needed for this -- we only inspect its fixed, hand-parameterized Gabor
# filters) and visualize a few of its actual kernels.
import matplotlib.pyplot as plt

_demo_model, _ = models.build_voneresnet50(pretrained=False, map_location="cpu")
_vone_block = models.get_vone_block(_demo_model)
print("VOneBlock channels:", _vone_block.out_channels,
      "(simple:", _vone_block.simple_channels, "+ complex:", _vone_block.complex_channels, ")")

# simple_conv_q0.weight has shape [out_channels, in_channels, ksize, ksize]; each output
# channel's kernel lives in exactly one input (color) channel (see modules.py::GFB.initialize,
# `random_channel`). Show 8 kernels spanning simple -> complex channel indices.
kernels = _vone_block.simple_conv_q0.weight.detach()
show_idx = [0, 8, 16, 32, 64, 128, 129, 200]
fig, axes = plt.subplots(1, len(show_idx), figsize=(2 * len(show_idx), 2))
for ax, idx in zip(axes, show_idx):
    ch = kernels[idx].abs().sum(dim=0)  # collapse the single populated input channel
    ax.imshow(ch, cmap="gray")
    kind = "simple" if idx < _vone_block.simple_channels else "complex"
    ax.set_title(f"ch {idx}\n({kind})", fontsize=9)
    ax.axis("off")
fig.suptitle("Real Gabor kernels from VOneBlock.simple_conv_q0 (q0 phase)")
common.save_figure(fig, os.path.join(RESULTS_DIR, "01_gabor_kernels.png"))
plt.show()


In [ ]:
# Concretely trace Stage 2 (simple/complex) and Stage 3 (noise) on one random image.
torch.manual_seed(CFG["seed"])
x_demo = torch.rand(1, 3, CFG["image_size"], CFG["image_size"])

r = _vone_block.gabors_f(x_demo)  # Stage 2 output: concat(simple, complex), Eq. above
print(f"gabors_f output: shape={tuple(r.shape)}, "
      f"mean={r.mean().item():.4f}, min={r.min().item():.4f} (>=0 as expected from ReLU/sqrt)")

_vone_block.set_noise_mode("neuronal", noise_scale=0.35, noise_level=0.07)
_vone_block.unfix_noise()
r_noisy_a = _vone_block.noise_f(r.clone())
r_noisy_b = _vone_block.noise_f(r.clone())
print(f"noise_f is stochastic when unfixed: identical calls differ by "
      f"{(r_noisy_a - r_noisy_b).abs().mean().item():.4f} on average (>0 expected).")

_vone_block.fix_noise(batch_size=1, seed=CFG["seed"])
r_fixed_a = _vone_block.noise_f(r.clone())
r_fixed_b = _vone_block.noise_f(r.clone())
print(f"noise_f is deterministic when fixed: identical calls differ by "
      f"{(r_fixed_a - r_fixed_b).abs().mean().item():.6f} on average (0 expected).")
assert torch.allclose(r_fixed_a, r_fixed_b), "fix_noise() should make noise_f deterministic."
_vone_block.unfix_noise()


## A necessary override: VOneBlock's input-gradient bug

Every attack in Math core 2 below needs $\nabla_x \ell(f(x), y)$ -- a gradient with
respect to the *input pixels*, not just the model's parameters. Computing that gradient
through the unmodified `external/vonenet/vonenet/modules.py::VOneBlock.gabors_f` raises
a `RuntimeError`.

**Root cause** (found with `torch.autograd.set_detect_anomaly(True)`, pinned to the
`PowBackward0` node inside the complex-cell energy computation): `gabors_f` computes the
complex-cell response from one channel-slice *view* of `s_q0`, then applies an
**in-place** ReLU (`nn.ReLU(inplace=True)`) to a *different*, disjoint channel-slice view
of that same `s_q0` tensor for the simple-cell response. PyTorch tracks in-place-mutation
version counters per underlying **storage**, not per region, so the later in-place ReLU
invalidates the earlier slice's saved-for-backward values even though the two slices
never physically overlap. This never surfaces during ordinary training (which only needs
gradients w.r.t. parameters), which is presumably why the upstream code never hit it.

Per `TEKNIK_REHBER.md` Sec. 3, `external/` is never edited directly. The fix
(`src/overrides.py::apply_vone_block_input_gradient_fix`) monkey-patches
`VOneBlock.gabors_f` at the class level with a version that `.clone()`s the simple-cell
slice before the in-place ReLU, giving it independent storage. This is verified below to
be **behaviorally identical on the forward pass** (deterministic-noise outputs compared
bit-for-bit) and to make the previously-failing gradient computation succeed.


In [ ]:
import src.overrides as overrides

# Sanity-check the patch's effect is exactly what it claims: identical forward output,
# input-gradient now computable. (Full derivation + anomaly-detection trace are in
# src/overrides.py's module docstring.)
_check_model, _ = models.build_voneresnet50(pretrained=False, map_location="cpu")
_check_model.eval()
_check_vb = models.get_vone_block(_check_model)
_check_x = torch.rand(1, 3, CFG["image_size"], CFG["image_size"])
_check_y = torch.tensor([0])

_check_vb.fix_noise(batch_size=1, seed=CFG["seed"])
with torch.no_grad():
    _logits_before = _check_model(_check_x.clone())

overrides.apply_vone_block_input_gradient_fix()
assert overrides.is_vone_block_input_gradient_fix_applied()

with torch.no_grad():
    _logits_after = _check_model(_check_x.clone())
assert torch.allclose(_logits_before, _logits_after, atol=1e-6), "Patch changed forward output!"
print(f"Forward output unchanged by the patch (max diff = "
      f"{(_logits_before - _logits_after).abs().max().item():.2e}).")

_check_vb.unfix_noise()
import torch.nn.functional as _F
_x_grad = _check_x.clone().requires_grad_(True)
_loss = _F.cross_entropy(_check_model(_x_grad), _check_y)
_grad, = torch.autograd.grad(_loss, _x_grad)
print(f"Input-gradient now computable: grad shape={tuple(_grad.shape)}, "
      f"norm={_grad.norm().item():.4f}.")
del _check_model, _check_vb, _check_x, _check_y, _logits_before, _logits_after, _x_grad, _loss, _grad


## Math core 2: PGD attack and Expectation-over-Transformation (EOT)

Robustness is measured with an $L_\infty$-bounded Projected Gradient Descent attack
(`src/eval_harness.py::pgd_attack`):

$$
x_{t+1} \;=\; \Pi_{\|\delta\|_\infty \le \varepsilon}\big(x_t + \alpha\,\mathrm{sign}(\bar g)\big),
\qquad x_0 = x,
$$

projecting back into the $\varepsilon$-ball around the original image and into valid
pixel range $[0, 1]$ at every step. For a **stochastic** model — VOneResNet-50 (B2) is
the only one here — a single forward/backward pass gives a high-variance, systematically
weak gradient estimate (a form of gradient masking that would make the model look more
robust than it is). Expectation-over-Transformation fixes this by averaging the gradient
over $N$ independent noise draws before taking the sign step:

$$
\bar g \;=\; \frac{1}{N}\sum_{i=1}^{N} \nabla_x\, \ell\big(f_{\xi_i}(x_t),\, y\big),
\qquad \xi_i \stackrel{\text{iid}}{\sim} \mathcal N(0, 1).
$$

`pgd_attack(..., eot_samples=N)` calls the model $N$ times per step with **fresh**
noise each time (i.e. `unfix_noise()`, never `fix_noise()`, during the attack) and
averages the resulting gradients — the opposite regime from the deterministic-eval
setup in Math core 1. B0/B1/B3 are deterministic, so `eot_samples=1` is exact for them;
B2 uses `eot_samples = CFG['eot_samples']`.


## Math core 3: why ResNet-50 as the ventral-stream backbone

This thesis uses ResNet-50 (not CORnet-S) as the main back-end for its proposed
architecture, with CORnet-S kept as a recurrent reference baseline (B3) for comparison.
The justification (docx Sec. 7, point 3; Liao & Poggio, 2016) is that a residual block's
update rule,

$$
h_{l+1} = h_l + \mathcal F(h_l),
$$

has the same additive-update functional form as one time-step of an unrolled recurrent
network, $h_{t+1} = h_t + \mathcal F(h_t)$. Under this reading, a feedforward ResNet with
$L$ blocks is architecturally analogous to $L$ steps of a (weight-untied) recurrent
computation — a biologically-motivated argument for treating ResNet-50 as a plausible
feedforward model of the ventral stream, onto which an explicit IT-feedback loop (added
in `04_it_feedback_multiglance.ipynb`) can be cleanly isolated as an *additional*,
ablatable source of recurrence, distinct from CORnet-S's built-in lateral/temporal
recurrence. This is cited as the field's argument, not re-derived here.


## Loading the four baselines

All four are frozen, pretrained (or, under `smoke_test`, randomly initialized)
classifiers -- no training happens in this notebook. `src/models.py::get_baseline`
returns `(model, normalization)`; the normalization tells every downstream cell which
`datasets.build_transform` convention to use for that specific model.


In [ ]:
BASELINE_NAMES = {
    "B0_resnet50": "resnet50",
    "B1_alexnet": "alexnet",
    "B2_voneresnet50": "voneresnet50",
    "B3_cornet_s": "cornet_s",
}

BASELINES = {}
for label, key in BASELINE_NAMES.items():
    model, normalization = models.get_baseline(key, pretrained=CFG["pretrained"])
    n_params = sum(p.numel() for p in model.parameters())
    BASELINES[label] = {"model": model.to(DEVICE).eval(), "normalization": normalization}
    print(f"{label:20s} normalization={normalization:9s} params={n_params/1e6:6.1f}M "
          f"pretrained={CFG['pretrained']}")


In [ ]:
# Deterministic evaluation for the stochastic baseline (B2): fix the VOneBlock noise
# for clean-accuracy scoring, matching TEKNIK_REHBER.md Sec. 4's reproducibility rule.
# (Unfixed again just before the PGD/EOT cell below, which needs fresh noise per sample.)
_b2_vone_block = models.get_vone_block(BASELINES["B2_voneresnet50"]["model"])
_b2_vone_block.fix_noise(batch_size=CFG["batch_size"], seed=CFG["seed"])
print("B2 VOneBlock noise fixed for deterministic clean-accuracy evaluation.")


## Clean-accuracy slice

All four baselines are 1000-way ImageNet classifiers, so this evaluation uses
ImageNet-1K validation images with their real ImageNet labels (`datasets.get_imagenet`)
-- **not** CIFAR-10, whose 10-class label space is not comparable to these models'
output layer. Under `smoke_test`, the same synthetic 1000-class stand-in used in
`00_setup_and_data.ipynb` is used here instead, purely to exercise the evaluation loop.


In [ ]:
clean_results = {}
for label, entry in BASELINES.items():
    ds = datasets.get_imagenet(
        DATA_DIR, split="val", normalization=entry["normalization"],
        image_size=CFG["image_size"], smoke_test=CFG["smoke_test"],
    )
    acc = eval_harness.evaluate_clean_accuracy(
        entry["model"], ds, device=DEVICE,
        batch_size=CFG["batch_size"], max_samples=CFG["max_samples"],
    )
    clean_results[label] = acc
    print(f"{label:20s} clean top-1 = {acc:.4f}"
          + ("  (smoke_test: synthetic data, not meaningful)" if CFG["smoke_test"] else ""))


## PGD robustness slice (with EOT for B2)

Each model is wrapped with `models.wrap_for_raw_pixel_input` so the attack perturbs raw
$[0,1]$ pixels identically across baselines regardless of their differing input
normalization (Math core 2). B2's VOneBlock noise is explicitly **unfixed** first, so
`eot_samples > 1` actually draws fresh noise on each of its $N$ forward passes.


In [ ]:
_b2_vone_block.unfix_noise()  # EOT needs fresh noise per forward pass, not fixed noise

pgd_results = {}
for label, entry in BASELINES.items():
    raw_ds = datasets.get_imagenet(
        DATA_DIR, split="val", normalization="raw",
        image_size=CFG["image_size"], smoke_test=CFG["smoke_test"],
    )
    wrapped = models.wrap_for_raw_pixel_input(entry["model"], entry["normalization"]).to(DEVICE).eval()
    eot_n = CFG["eot_samples"] if label == "B2_voneresnet50" else 1
    acc = eval_harness.evaluate_pgd_robust_accuracy(
        wrapped, raw_ds, device=DEVICE,
        eps=CFG["pgd_eps"], alpha=CFG["pgd_alpha"], steps=CFG["pgd_steps"],
        eot_samples=eot_n, batch_size=CFG["batch_size"], max_samples=CFG["max_samples"],
    )
    pgd_results[label] = acc
    print(f"{label:20s} PGD(eps={CFG['pgd_eps']:.3f}, eot={eot_n:2d}) top-1 = {acc:.4f}"
          + ("  (smoke_test: synthetic data, not meaningful)" if CFG["smoke_test"] else ""))


## Comparison and handoff artifacts

The bar chart below reports, side by side, clean accuracy and PGD-robust accuracy for
each baseline. Under a full (non-smoke-test) run with real pretrained weights, the
expected qualitative pattern (Bulgu 1) is that B2 (VOneResNet-50) loses less accuracy
under PGD than B0 (plain ResNet-50) despite starting from comparable clean accuracy --
this is the reproduction target for Aşama 1, not a number this notebook asserts on its
own synthetic/smoke-test data.


In [ ]:
import numpy as np

labels = list(BASELINES.keys())
clean_vals = [clean_results[l] for l in labels]
pgd_vals = [pgd_results[l] for l in labels]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width / 2, clean_vals, width, label="clean")
ax.bar(x + width / 2, pgd_vals, width, label=f"PGD (eps={CFG['pgd_eps']:.3f})")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("top-1 accuracy")
title = "Baseline comparison (B0-B3)"
if CFG["smoke_test"]:
    title += " -- SMOKE TEST: synthetic data, not real numbers"
ax.set_title(title)
ax.legend()
common.save_figure(fig, os.path.join(RESULTS_DIR, "01_baseline_comparison.png"))
plt.show()

common.save_json(
    {"clean_accuracy": clean_results, "pgd_robust_accuracy": pgd_results, "cfg": CFG},
    os.path.join(RESULTS_DIR, "01_baseline_metrics.json"),
)


In [ ]:
env_stamp = common.environment_stamp(PROJECT_ROOT)
env_stamp["cfg"] = CFG
common.save_json(env_stamp, os.path.join(RESULTS_DIR, "01_environment.json"))

common.save_checkpoint(
    {"baselines_evaluated": list(BASELINES.keys()), "cfg": CFG,
     "timestamp_utc": env_stamp["timestamp_utc"]},
    os.path.join(CKPT_DIR, "01_baselines_evaluated.pt"),
)
print("Saved results/01_baseline_metrics.json, results/01_environment.json, "
      "checkpoints/01_baselines_evaluated.pt")


## Summary and handoff

Reproduced so far:
- B0–B3 loaded through a uniform `(model, normalization)` interface (`src/models.py`).
- VOneBlock's LNP math (Gabor bank, simple/complex cells, neuronal stochasticity) traced
  against the real `external/vonenet` code and visualized.
- A PGD-with-EOT attack (`src/eval_harness.py`), verified independently (closed-form
  gradient match, $\varepsilon$-ball / $[0,1]$ clamping, correct EOT call count) before
  being used here.
- One robustness slice (clean + PGD accuracy) computed for all four baselines and saved
  for later comparison against the full model's ablations.

**Next:** `02_foveation_rblur_and_periphery.ipynb` — introduces the foveal front-end
(R-Blur reimplementation), the three periphery regimes (flat blur / i.i.d. noise /
trace-based metameric noise), and the Watson-(2014)-grounded SNR(r) formalization that
is this thesis's original contribution (see `TRACE_BASED_NOISE_REHBERI.md`).
